# MotorCam – fáze 4+5: demo video a export do telefonu
1. Demo: nahrané video + CSV senzorů → video s overlayem, minimapou, kontrolkou a pípáním.
2. Export modelů do TFLite a test rychlosti. Soubory `models/export/*.tflite` pak nahraj do aplikace
   (☰ → Načíst model…).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE_DIR = '/content/drive/MyDrive/motorcam'
REPO_URL = 'https://github.com/MiniGue1/motorcam.git'   # u soukromého repa: https://<token>@github.com/...

import os
%cd /content
if not os.path.exists('motorcam'):
    !git clone -q $REPO_URL motorcam
%cd /content/motorcam
!pip install -q -r requirements.txt
# archivy, vlastní data, modely a výsledky trvale na Google Drive
for d in ['data/raw/rdd2022/archives', 'data/raw/rscd/archives', 'data/custom', 'models', 'results']:
    os.makedirs(f'{DRIVE_DIR}/{d}', exist_ok=True); os.makedirs(os.path.dirname(d) or '.', exist_ok=True)
    if not os.path.islink(d):
        !rm -rf $d
        os.symlink(f'{DRIVE_DIR}/{d}', d)
import torch; print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NENÍ – Runtime → Change runtime type → T4 GPU')

In [ ]:
!pip install -q tensorflow-cpu onnx2tf onnxslim onnx_graphsurgeon sng4onnx tf_keras ai-edge-litert onnxruntime imageio-ffmpeg

## Export do TFLite + kontrola formátu pro aplikaci

In [ ]:
!python -m src.demo.export_tflite surface models/surface_finetuned/best.pt || python -m src.demo.export_tflite surface models/surface_public/best.pt
!python -m src.demo.export_tflite detector models/det_finetuned/weights/best.pt || python -m src.demo.export_tflite detector models/det_public/weights/best.pt
!python -m src.demo.export_tflite bench models/export/surface.tflite
!python -m src.demo.export_tflite bench models/export/detector.tflite
import pandas as pd; pd.read_csv('results/tables/tflite_rychlost.csv')

## Demo video
Video a CSV z aplikace MotorCam (tlačítko REC) nahraj na Drive do `motorcam/data/custom/videos/`.
CSV ze Sensor Loggeru: přidej `--offset <s>` (čas senzoru, kdy začalo video).

In [ ]:
VIDEO = 'data/custom/videos/jizda_20261003_101500.mp4'   # uprav
SENSORS = VIDEO.replace('.mp4', '.csv')
DET = 'models/det_finetuned/weights/best.pt'
SURF = 'models/surface_finetuned/best.pt'
!python -m src.demo.render "$VIDEO" --sensors "$SENSORS" --det "$DET" --surface "$SURF" \
    --device cuda --every 1 --out results/demo/demo.mp4

In [ ]:
from IPython.display import HTML
from base64 import b64encode
mp4 = open('results/demo/demo.mp4', 'rb').read()
HTML(f'<video width=800 controls src="data:video/mp4;base64,{b64encode(mp4).decode()}"></video>')